In [ ]:
# ==========================================================
# CELL 1: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

In [ ]:
# ==========================================================
# CELL 2: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

In [ ]:
# ==========================================================
# CELL 2.5: IMPORT UMUM — dipakai di banyak cell berikutnya
# ==========================================================
import os
import pandas as pd
import numpy as np

In [ ]:
# ==========================================================
# CELL 2.6 (BARU): STATUS PIPELINE -- CEK SUDAH SAMPAI MANA
# ==========================================================
# Jalankan kapan saja untuk lihat progres tanpa harus scroll ke atas.
# ==========================================================
from src import config
import os

def cek_status():
    config.set_data_version("v2")
    config.set_proxy(3)  # proxy final

    steps = {
        "1. Data mentah (scraping)": config.RAW_DATA_FILE,
        "2. Data v2 preprocessed": config.TRAIN_RAW_FILE,
        "3. Tabel ablasi proxy (pilot study)": config.PROXY_QUALITY_LOG_FILE,
        "4. Cleaned data (proxy final)": config.TRAIN_CLEANED_HARD_FILE,
        "5. Sample validasi manusia": config.HUMAN_VALIDATION_FILE,
        "6. Hasil validasi manusia (terisi)": config.HUMAN_VALIDATION_RESULT_FILE,
        "7. Progress training M1-M6": config.PROGRESS_FILE,
        "8. Hasil 6 skenario": config.FINAL_RESULTS_TABLE_FILE,
        "9. Uji signifikansi": config.SIGNIFICANCE_TEST_FILE,
    }
    print("📋 STATUS PIPELINE (proxy final: finetuned_corn, data v2)\n")
    for label, path in steps.items():
        status = "✅" if os.path.exists(path) else "⬜"
        print(f"{status} {label}")

cek_status()

In [ ]:
# ==========================================================
# CELL 3: SCRAPING GOOGLE PLAY STORE
# ==========================================================
# ⚠️ JALANKAN SEKALI SAJA SEUMUR PROYEK. RAW_DATA_FILE TIDAK versioned
# (sumber mentah tunggal untuk semua DATA_VERSION) -- jangan dijalankan
# ulang setelah selesai.
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

In [ ]:
# ==========================================================
# CELL 3.5 (BARU): MIGRASI DATA LAMA -> DATA_VERSION="v1"
# ==========================================================
# File preprocessing lama (sebelum fix kamus slang + dedup konflik) masih
# ada di Drive dengan nama TANPA suffix versi (reviews_clean.csv,
# split_train_raw.csv, split_test.csv). Supaya bisa dipakai sebagai
# pembanding "v1" di ablasi Cell 6.6 nanti, kita salin (bukan pindah)
# ke nama baru yang sesuai skema DATA_VERSION.
#
# JALANKAN SEKALI SAJA. Kalau file lama tidak ada (proyek baru dari nol,
# belum pernah preprocessing sebelumnya), cell ini otomatis dilewati --
# artinya ablasi v1-vs-v2 di Cell 6.6 tidak relevan untukmu, langsung
# lanjut pakai v2 saja.
# ==========================================================
import shutil
from src import config

_OLD_CLEAN = os.path.join(config.DATA_PROCESSED_DIR, "reviews_clean.csv")
_OLD_TRAIN = os.path.join(config.DATA_PROCESSED_DIR, "split_train_raw.csv")
_OLD_TEST  = os.path.join(config.DATA_PROCESSED_DIR, "split_test.csv")

config.set_data_version("v1")
_migrated = 0
for old_path, new_path in [
    (_OLD_CLEAN, config.CLEAN_TEXT_FILE),
    (_OLD_TRAIN, config.TRAIN_RAW_FILE),
    (_OLD_TEST, config.TEST_FILE),
]:
    if os.path.exists(old_path) and not os.path.exists(new_path):
        shutil.copy(old_path, new_path)
        print(f"📋 Disalin: {old_path} -> {new_path}")
        _migrated += 1
    elif os.path.exists(new_path):
        print(f"✅ Sudah ada: {new_path}")
    else:
        print(f"⚠️ Tidak ditemukan (dilewati): {old_path}")

if _migrated == 0 and not os.path.exists(config.TRAIN_RAW_FILE):
    print("\nℹ️  Tidak ada data v1 untuk dimigrasikan -- proyek dimulai dari nol.")
    print("   Ablasi v1-vs-v2 (Cell 6.6) tidak relevan, lanjut langsung ke v2.")

config.set_data_version("v2")  # kembalikan ke default kerja

In [ ]:
# ==========================================================
# CELL 3.6 (BARU): MIGRASI VALIDASI MANUSIA YANG SUDAH DIISI -> v1
# ==========================================================
# Kerja manual mengisi human_verdict untuk 50 sample JANGAN sampai hilang
# cuma karena skema penamaan file berubah. Sesuaikan proxy_name di bawah
# kalau validasi manusia lama kamu itu untuk proxy selain P4.
import shutil
from src import config

_OLD_HV_SAMPLE = os.path.join(config.HUMAN_VALIDATION_DIR, "human_validation_sample.csv")
_OLD_HV_RESULT = os.path.join(config.HUMAN_VALIDATION_DIR, "human_validation_result.csv")

config.set_data_version("v1")
config.set_proxy(3)  # ganti kalau validasi manusia lama itu untuk proxy lain

for old_path, new_path in [
    (_OLD_HV_SAMPLE, config.HUMAN_VALIDATION_FILE),
    (_OLD_HV_RESULT, config.HUMAN_VALIDATION_RESULT_FILE),
]:
    if os.path.exists(old_path) and not os.path.exists(new_path):
        shutil.copy(old_path, new_path)
        print(f"📋 Disalin: {old_path} -> {new_path}")
    elif os.path.exists(new_path):
        print(f"✅ Sudah ada: {new_path}")
    else:
        print(f"⚠️ Tidak ditemukan: {old_path}")

config.set_data_version("v2")  # kembalikan ke default kerja

In [ ]:
# ==========================================================
# CELL 3.7 (BARU): RESET PAKSA FILE v2 -- JALANKAN HANYA SEKALI
# ==========================================================
# ⚠️ Cell ini MENGHAPUS hasil preprocessing v2 yang ada, memaksa Cell 4
# membangunnya ulang dari nol. HANYA jalankan ini kalau kamu SENGAJA mau
# rebuild v2 dari awal (mis. setelah ubah logika preprocessing). JANGAN
# jalankan tanpa sadar di run rutin -- ini TIDAK menyentuh data v1,
# raw data, atau human_validation, jadi aman dari sisi itu.
# ==========================================================
import os
from src import config
config.set_data_version("v2")
for f in [config.CLEAN_TEXT_FILE, config.TRAIN_RAW_FILE, config.TEST_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ Dihapus: {f}")

In [ ]:
# ==========================================================
# CELL 4: PREPROCESSING v2 -- fix kamus slang + dedup konflik teks-rating
# ==========================================================
from sklearn.model_selection import train_test_split
from src.preprocess import run_preprocessing
from src import config

config.set_data_version("v2")

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.TEST_FILE):
    print(f"✅ Split {config.DATA_VERSION} sudah ada -- preprocessing dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} baris | Test: {len(df_test)} baris")
else:
    print("=" * 60)
    print(f" PREPROCESSING ({config.DATA_VERSION}) ")
    print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) ")
    print("=" * 60)
    df_train, df_test = train_test_split(
        df_clean, test_size=0.2, random_state=42, stratify=df_clean["rating"]
    )
    df_train.to_csv(config.TRAIN_RAW_FILE, index=False)
    df_test.to_csv(config.TEST_FILE, index=False)

    print(f"✅ Train: {len(df_train)} baris -> {config.TRAIN_RAW_FILE}")
    print(f"✅ Test : {len(df_test)} baris -> {config.TEST_FILE}")

In [ ]:
# ==========================================================
# CELL 4.5 (BARU): BANDINGKAN UKURAN & CAKUPAN SLANG v1 vs v2
# ==========================================================
from src import config

_summary_v1 = os.path.join(config.RESULTS_DIR, "preprocessing_summary__v1.csv")
_summary_v2 = os.path.join(config.RESULTS_DIR, "preprocessing_summary__v2.csv")

print("📊 Perbandingan preprocessing v1 (lama, ada bug) vs v2 (sudah di-fix):\n")
if os.path.exists(_summary_v1):
    display(pd.read_csv(_summary_v1))
else:
    print("   (ringkasan v1 tidak ada -- kemungkinan proyek dimulai dari nol)")

if os.path.exists(_summary_v2):
    display(pd.read_csv(_summary_v2))
else:
    print("   ⚠️ Ringkasan v2 belum ada -- pastikan Cell 4 sudah dijalankan.")

In [ ]:
# ==========================================================
# CELL 5: PILOT STUDY -- ABLASI PROXY 0-3 (data v2)
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")

PILOT_PROXY_IDS = [0, 1, 2, 3]  # 4 (fusion) & 6 (IndoBERTweet) dijalankan terpisah di cell lain

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} | DATA = {config.DATA_VERSION} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table[pilot_table["data_version"] == "v2"])

In [ ]:
# ==========================================================
# CELL 6: PROXY FINAL SEMENTARA (P4, CORN) -- data v2
# ==========================================================
# ⚠️ RESTART RUNTIME dulu sebelum cell ini kalau tadi jalankan Cell 5
# (loop pilot study), supaya config bersih.
# ==========================================================
from src import config

config.set_data_version("v2")
config.set_proxy(3)
print(f"📌 Proxy sementara: [{config.PROXY_ID}] {config.PROXY_NAME} | data {config.DATA_VERSION}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

In [ ]:
# ==========================================================
# CELL 6.5: CEK KELENGKAPAN FILE DI DRIVE
# ==========================================================
from src import config
import os

checks = {
    "Data train (v2)": config.TRAIN_RAW_FILE,
    "Data test (v2)": config.TEST_FILE,
    "Tabel ablasi proxy": config.PROXY_QUALITY_LOG_FILE,
}

all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {label}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ Semua file ditemukan -- aman lanjut ke Cell 6.6.")
else:
    print("\n⛔ Ada file tidak ditemukan! Cek akun Drive yang dipakai.")

In [ ]:
# ==========================================================
# CELL 6.6 (BARU): ABLASI EFEK FIX DATA -- P4 di v1 vs v2
# ==========================================================
# Satu variabel yang berubah: DATA_VERSION. Metode proxy tetap sama (P4).
# Ini mengukur murni efek fix kamus slang + dedup konflik teks-rating,
# terpisah dari eksperimen backbone (Cell 6.7 & 6.8).
#
# CATATAN: karena cache OOF proxy 3 di v1 belum pernah dihitung dengan
# nama file baru (oof_pred_probs__finetuned_corn__v1.npy), cell ini akan
# fine-tune ulang dari nol untuk v1 (bukan dari cache) -- wajar, sekali
# saja, dan hasilnya deterministik (seed=42) sehingga sebanding dengan
# angka lama yang sudah kamu punya di laporan sebelumnya.
# ==========================================================
from src import config
from src.clean import run_confident_learning

if not os.path.exists(os.path.join(config.DATA_PROCESSED_DIR, "split_train_raw__v1.csv")):
    print("ℹ️  Data v1 tidak tersedia (lihat Cell 3.5) -- ablasi ini dilewati.")
else:
    config.set_data_version("v1")
    config.set_proxy(3)
    print(f"\n{'='*70}\n P4 DI DATA v1 (sebelum fix) \n{'='*70}")
    df_noise_v1, metrics_v1 = run_confident_learning()

    config.set_data_version("v2")
    config.set_proxy(3)
    print(f"\n{'='*70}\n P4 DI DATA v2 (sesudah fix) \n{'='*70}")
    df_noise_v2, metrics_v2 = run_confident_learning()

    print("\n" + "=" * 60)
    print(" PERBANDINGAN EFEK FIX DATA (P4, proxy sama) ")
    print("=" * 60)
    print(f"v1 (sebelum fix): Acc={metrics_v1['accuracy']:.4f} | MAE={metrics_v1['mae']:.4f} "
          f"| Off-by-1={metrics_v1['off_by_one']:.4f} | QWK={metrics_v1['qwk']:.4f}")
    print(f"v2 (sesudah fix): Acc={metrics_v2['accuracy']:.4f} | MAE={metrics_v2['mae']:.4f} "
          f"| Off-by-1={metrics_v2['off_by_one']:.4f} | QWK={metrics_v2['qwk']:.4f}")

    # kembalikan ke v2 -- default kerja untuk cell selanjutnya
    config.set_data_version("v2")
    config.set_proxy(3)

In [ ]:
# ==========================================================
# CELL 6.7: ABLASI TAMBAHAN -- P5 (FUSION SENTIMEN), data v2
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")
config.set_proxy(4)
try:
    df_noise_p5, proxy_metrics_p5 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 4 (fusion) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke sementara: [{config.PROXY_ID}] {config.PROXY_NAME}")

In [ ]:
# ==========================================================
# CELL 6.8 (BARU): ABLASI BACKBONE -- P6 (IndoBERTweet + CORN), data v2
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")
config.set_proxy(6)
print(f"📌 Proxy: [{config.PROXY_ID}] {config.PROXY_NAME} | Backbone: {config.PRETRAINED_MODEL_NAME}")
try:
    df_noise_p6, proxy_metrics_p6 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 6 (IndoBERTweet) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke sementara: [{config.PROXY_ID}] {config.PROXY_NAME}")

In [ ]:
# ==========================================================
# CELL 6.9 (BARU): RINGKASAN SEMUA PROXY DI DATA v2 -- PILIH FINAL DI SINI
# ==========================================================
from src import config

table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
table_v2 = table[table["data_version"] == "v2"].sort_values("qwk", ascending=False)
print("📊 Semua proxy yang sudah diuji di data v2, diurutkan dari QWK tertinggi:")
display(table_v2)

print("\n⚠️  TENTUKAN PROXY_ID FINAL secara manual berdasarkan tabel di atas,")
print("   lalu isi di FINAL_PROXY_ID di bawah sebelum lanjut ke Cell 7.")

In [ ]:
# ==========================================================
# CELL 7: KUNCI PROXY FINAL + VALIDASI MANUSIA — WAJIB SEBELUM LANJUT
# ==========================================================
# GANTI angka ini sesuai proxy yang kamu pilih dari tabel Cell 6.9
FINAL_PROXY_ID = 3   # <-- EDIT DI SINI

from src import config
config.set_data_version("v2")
config.set_proxy(FINAL_PROXY_ID)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME} | data {config.DATA_VERSION}")

print("\n⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan Cell 7.5 di bawah untuk cek kelengkapan + hitung agreement rate.")

In [ ]:
# ==========================================================
# CELL 7.5: HITUNG AGREEMENT VALIDASI MANUSIA
# ==========================================================
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 8 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

In [ ]:
# ==========================================================
# CELL 7.6 (BARU): DIAGNOSTIK -- CONFUSION MATRIX + F1 PER KELAS PROXY FINAL
# ==========================================================
# Versi resmi di pipeline final (data v2, proxy terkunci) dari diagnostik
# yang sebelumnya cuma ada di notebook terpisah (SKRIPSITEST, data v1 lama).
# Murah -- pakai cache OOF pred_probs yang sudah ada, tidak retrain.
# ==========================================================
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from src.proxy import get_proxy_pred_probs

df_train_diag = pd.read_csv(config.TRAIN_RAW_FILE)
labels_diag = df_train_diag["rating"].values - 1
texts_diag = df_train_diag["cleaned_text"].tolist()

pred_probs_diag = get_proxy_pred_probs(texts_diag, labels_diag)
preds_diag = np.argmax(pred_probs_diag, axis=1)

print(f"📊 Classification report -- proxy [{config.PROXY_NAME}], data {config.DATA_VERSION}:\n")
report_str = classification_report(
    labels_diag, preds_diag,
    target_names=[f"Bintang {i+1}" for i in range(5)], digits=3,
)
print(report_str)

cm_diag = confusion_matrix(labels_diag, preds_diag)
cm_df_diag = pd.DataFrame(
    cm_diag,
    index=[f"Asli {i+1}" for i in range(5)],
    columns=[f"Pred {i+1}" for i in range(5)],
)
display(cm_df_diag)

# simpan supaya tidak perlu hitung ulang untuk Bab IV
report_path = os.path.join(config.RESULTS_DIR, f"proxy_classification_report__{config.PROXY_NAME}__{config.DATA_VERSION}.txt")
with open(report_path, "w") as f:
    f.write(report_str)
cm_df_diag.to_csv(os.path.join(config.RESULTS_DIR, f"proxy_confusion_matrix__{config.PROXY_NAME}__{config.DATA_VERSION}.csv"))
print(f"\n💾 Tersimpan -> {report_path}")

In [ ]:
# ==========================================================
# CELL 8: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

# Konfirmasi eksplisit -- cegah salah backbone/data tanpa sadar
print(f"🔧 Training M1-M6 akan pakai backbone: {config.PRETRAINED_MODEL_NAME} | "
      f"data: {config.DATA_VERSION} | proxy aktif: {config.PROXY_NAME}")

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(f" 🏆 HASIL 6 SKENARIO -- backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION} 🏆")
print("=" * 100)
display(df_final)

In [ ]:
# ==========================================================
# CELL 9: UJI SIGNIFIKANSI -- WILCOXON + HOLM-BONFERRONI
# ==========================================================
from src.significance import (
    collect_all_predictions,
    aggregate_errors_across_seeds,
    run_significance_test,
    run_all_effect_sizes,
)
from src import config

print(f"🔧 Menguji hasil untuk backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION}")

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

print("\n📊 Mengagregasi absolute error across seed...")
aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)

effect_sizes.to_csv(os.path.join(config.RESULTS_DIR, f"effect_sizes__{config.PROXY_NAME}__{config.DATA_VERSION}.csv"), index=False)

In [ ]:
# ==========================================================
# CELL 9.5 (BARU): BREAKDOWN ERROR PER KELAS RATING -- M4 vs M5 vs M6
# ==========================================================
# Menjawab: apakah M5/M6 (hasil cleaning) memburuk KHUSUSNYA di kelas
# rating tengah (bintang 2/3/4), atau merata di semua kelas? Kalau khusus
# di kelas tengah, itu mendukung argumen "cleaning membuang contoh
# sulit/ambigu di kelas tengah", bukan random noise yang tersebar merata.
#
# Pakai ulang true_labels & preds_per_seed dari Cell 9 kalau sudah ada di
# sesi ini -- kalau belum, panggil collect_all_predictions() lagi (cepat,
# cuma forward pass dari checkpoint, TIDAK retrain).
# ==========================================================
import numpy as np
import pandas as pd
from src.significance import collect_all_predictions
from src import config

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if "true_labels" not in dir() or "preds_per_seed" not in dir():
    print("📥 true_labels/preds_per_seed belum ada di sesi ini, memuat ulang...")
    true_labels, preds_per_seed = collect_all_predictions(scenarios)
else:
    print("⚡ Memakai true_labels/preds_per_seed yang sudah ada dari Cell 9.")

TARGET_SCENARIOS = ["M4_Baseline_CORN", "M5_CleanedHard_CORN", "M6_CleanedSevere_CORN"]

def per_class_breakdown(true_labels, preds_per_seed, scenario_names):
    """MAE & accuracy per kelas rating asli (1-5), dirata-ratakan across 3 seed."""
    rows = []
    for name in scenario_names:
        preds_stack = np.stack(list(preds_per_seed[name].values()))  # (3, n_test)
        for rating in sorted(np.unique(true_labels)):
            mask = true_labels == rating
            mae_per_seed, acc_per_seed = [], []
            for preds in preds_stack:
                errs = np.abs(true_labels[mask] - preds[mask])
                mae_per_seed.append(errs.mean())
                acc_per_seed.append((preds[mask] == true_labels[mask]).mean())
            rows.append({
                "scenario": name, "rating_asli": rating, "n_test": int(mask.sum()),
                "mae_mean": np.mean(mae_per_seed), "mae_std": np.std(mae_per_seed),
                "acc_mean": np.mean(acc_per_seed),
            })
    return pd.DataFrame(rows)

breakdown_df = per_class_breakdown(true_labels, preds_per_seed, TARGET_SCENARIOS)

print("📊 MAE per kelas rating asli, per skenario (rata-rata 3 seed):\n")
pivot_mae = breakdown_df.pivot(index="rating_asli", columns="scenario", values="mae_mean")[TARGET_SCENARIOS]
display(pivot_mae.round(4))

print("\n📊 Selisih MAE vs M4 -- POSITIF = cleaning MEMBUAT LEBIH BURUK di kelas itu:\n")
diff_df = pd.DataFrame({
    "M5_minus_M4": pivot_mae["M5_CleanedHard_CORN"] - pivot_mae["M4_Baseline_CORN"],
    "M6_minus_M4": pivot_mae["M6_CleanedSevere_CORN"] - pivot_mae["M4_Baseline_CORN"],
})
display(diff_df.round(4))

out_path = os.path.join(config.RESULTS_DIR, f"per_class_breakdown__{config.PROXY_NAME}__{config.DATA_VERSION}.csv")
breakdown_df.to_csv(out_path, index=False)
print(f"\n💾 Tersimpan -> {out_path}")

In [ ]:
# ==========================================================
# CELL 9.6 (BARU): CEK JUMLAH SAMPEL TEST PER KELAS RATING
# ==========================================================
# Memastikan selisih MAE di Cell 9.5 bukan artefak sampel kecil --
# semua kelas idealnya di atas ~250 baris supaya bisa dipercaya.
# ==========================================================
display(breakdown_df[breakdown_df["scenario"] == "M4_Baseline_CORN"][["rating_asli", "n_test"]])

In [ ]:
# ==========================================================
# CELL 10: RINGKASAN AKHIR -- SIAP DISALIN KE BAB 4
# ==========================================================
import pandas as pd
from src import config

print("=" * 70)
print(f" RINGKASAN LENGKAP -- backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION} ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (semua data_version) -- untuk Bab 1:")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    proxy_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
    display(proxy_table)
else:
    print("   ⚠️ Belum ada.")

print(f"\n[2] VALIDASI MANUSIA ({config.PROXY_NAME}, {config.DATA_VERSION}):")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 7.5 dulu.")

print(f"\n[3] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 8 dulu.")

print(f"\n[4] UJI SIGNIFIKANSI -- untuk Bab 4:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 9 dulu.")

print(f"\n[5] EFFECT SIZE + CI 95% -- untuk Bab 4:")
effect_size_file = os.path.join(config.RESULTS_DIR, f"effect_sizes__{config.PROXY_NAME}__{config.DATA_VERSION}.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 9 dulu.")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)